In [1]:
from pyspark.sql import functions as F

In [2]:
from spark_session import get_spark
import sources

In [3]:
spark = get_spark("aggregate")
orders = sources.orders(spark).filter("quantity > 0")
orders.groupBy("status").count().orderBy(F.desc("count")).show()

+--------+------+
|  status| count|
+--------+------+
|    paid|208807|
| shipped| 69086|
|refunded|  1286|
+--------+------+



In [5]:
daily = (
    orders
    .groupBy(F.to_date("order_ts").alias("day"))
    .agg(F.count("*").alias("orders"),
         F.sum("quantity").alias("units"),
         F.countDistinct("customer_id").alias("customers"),
         F.round(F.avg("quantity"), 2).alias("avg_qty")
         )
    .orderBy("day")
)
daily.show(5, truncate=False)

+----------+------+-----+---------+-------+
|day       |orders|units|customers|avg_qty|
+----------+------+-----+---------+-------+
|2026-10-01|20146 |40376|2000     |2.0    |
|2026-10-02|20052 |40065|1999     |2.0    |
|2026-10-03|20026 |39982|2000     |2.0    |
|2026-10-04|19993 |40080|2000     |2.0    |
|2026-10-05|19966 |40151|2000     |2.01   |
+----------+------+-----+---------+-------+
only showing top 5 rows


In [10]:
(
    orders.groupBy(F.to_date("order_ts").alias("day"))
    .pivot("status", ["paid", "shipped", "refunded"])
    .count().orderBy("day").show(3)
)

top = (
    orders.groupBy("product_id")
    .agg(F.sum("quantity").alias("units"))
    .orderBy(F.desc("units"))
    .limit(5)
)

top.show()
print("distinct customers: ", orders.select("customer_id").distinct().count(),
      "approx: ", orders.agg(F.approx_count_distinct("customer_id")).first()[0])

+----------+-----+-------+--------+
|       day| paid|shipped|refunded|
+----------+-----+-------+--------+
|2026-10-01|14849|   4981|     316|
|2026-10-02|14911|   4925|     216|
|2026-10-03|15010|   4847|     169|
+----------+-----+-------+--------+
only showing top 3 rows
+----------+-----+
|product_id|units|
+----------+-----+
|      P018|13948|
|      P028|13934|
|      P017|13887|
|      P013|13836|
|      P036|13819|
+----------+-----+

distinct customers:  2001 approx:  2023


In [ ]:
spark.stop()